# SAAM — Part II: Portfolio Allocation with Carbon Emission Reduction
**Region:** North America & Europe (AMER + EUR) &nbsp;|&nbsp; **Scope:** 1 & 2  
**Out-of-sample period:** 2014–2025 &nbsp;|&nbsp; **Rebalancing:** Annual

---

This notebook extends Part I with three climate-aware portfolio strategies:

| Section | Strategy | Symbol |
|---------|----------|---------|
| 3.1 | Carbon intensity metrics | — |
| 3.2 | MV portfolio with 50% CF reduction | $P^{(mv)}_{oos}(0.5)$ |
| 3.3 | Tracking-error minimisation with 50% CF reduction | $P^{(vw)}_{oos}(0.5)$ |
| 4.1 | Net-zero pathway (−10% CF per year) | $P^{(vw)}_{oos}(NZ)$ |

**Prerequisites:** Run `01_part1_portfolio_allocation.ipynb` first, or ensure
all Part I variables are available in the kernel.

## 0. Setup & Imports

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC_DIR = os.path.abspath(os.path.join('..', 'src'))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from io_utils        import load_ts, clean_prices, compute_returns, ffill_annual, filter_region
from portfolio_utils import (
    get_estimation_window, build_investment_set,
    pairwise_covariance, compute_mv_returns, compute_vw_returns, compute_stats,
)
from carbon_utils import (
    compute_carbon_intensity, compute_waci, compute_carbon_footprint,
    compute_vw_carbon_footprint, _footprint_constraint_vector,
    solve_mv_carbon_constrained, solve_te_carbon_constrained,
)
from plot_utils import plot_cumulative_returns, plot_carbon_metric

# ── Paths (same as Part I) ─────────────────────────────────────────────────
CONFIG_PATH = os.path.join('..', 'config', 'paths.json')
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        cfg = json.load(f)
    DATA_DIR    = cfg['data_dir']
    FIGURES_DIR = cfg['figures_dir']
    TABLES_DIR  = cfg['tables_dir']
    FILES       = cfg['files']
else:
    DATA_DIR    = os.path.join('..', 'data', 'raw', 'extracted') + os.sep
    FIGURES_DIR = os.path.join('..', 'results', 'figures') + os.sep
    TABLES_DIR  = os.path.join('..', 'results', 'tables') + os.sep
    FILES = {
        'static':   'Static_2025.xlsx',
        'ri_m':     'DS_RI_T_USD_M_2025.xlsx',
        'mv_m':     'DS_MV_T_USD_M_2025.xlsx',
        'mv_y':     'DS_MV_T_USD_Y_2025.xlsx',
        'co2_s1':   'DS_CO2_SCOPE_1_Y_2025.xlsx',
        'co2_s2':   'DS_CO2_SCOPE_2_Y_2025.xlsx',
        'rev':      'DS_REV_Y_2025.xlsx',
        'rf':       'Risk_Free_Rate_2025.xlsx',
    }

for d in [FIGURES_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

SCOPE    = '1+2'      # Scope 1 + 2 as assigned to our group
Y0       = 2013       # Base year for net-zero constraint
THETA    = 0.10       # Annual decarbonisation rate (10%)
V0       = 1_000_000  # Starting portfolio value ($1M)

print('✓ Setup complete')

## 1. Reload Part I Data

Re-run data loading and cleaning so this notebook is fully self-contained.

In [ ]:
# ── TODO: copy the data loading cells from Part I here ────────────────────
# This ensures the notebook runs top-to-bottom without requiring
# a shared kernel with notebook 01.
#
# Once Part II analysis is finalised, both parts can be merged into a
# single submission notebook.
raise NotImplementedError('Re-run data loading from Part I before proceeding.')

## 2. Carbon Intensity Computation

$$
CI_{i,Y} = \frac{E^{S1}_{i,Y} + E^{S2}_{i,Y}}{\text{Rev}_{i,Y} / 1000}
\quad [\text{tCO}_2\text{e / \$M revenue}]
$$

Note: Revenue is in thousands USD in the raw data → divide by 1 000 to get $M.

In [ ]:
ci = compute_carbon_intensity(co2_s1_f, co2_s2_f, rev_f, scope=SCOPE)

# Total CO2 (Scope 1 + 2) for carbon footprint computation
co2_total = co2_s1_f.add(co2_s2_f, fill_value=0)

print(f'Carbon intensity matrix: {ci.shape}')
print(f'Annual CI summary (2013–2024):')
print(ci[[y for y in range(2013, 2025) if y in ci.columns]].describe().to_string())

## 3. Carbon Metrics of the Part I Portfolios

### 3.1 WACI and CF of the MV and VW portfolios

$$
\text{WACI}^{(p)}_Y = \sum_i \alpha_{i,Y} \cdot CI_{i,Y}
\qquad
\text{CF}^{(p)}_Y = \frac{1}{V_Y} \sum_i o_{i,Y} \cdot E_{i,Y}
$$

In [ ]:
# TODO: compute WACI and CF for mv_weights and vw benchmark each year
# Hint: use compute_waci(), compute_carbon_footprint(), compute_vw_carbon_footprint()

years = list(range(2013, 2025))

waci_mv, cf_mv = [], []
waci_vw, cf_vw = [], []

portfolio_value = V0  # evolves each year — update after computing returns

for Y in years:
    isins = investment_sets[Y]

    # MV portfolio
    w_mv = mv_weights[Y]
    waci_mv.append(compute_waci(w_mv, ci, Y))
    cf_mv.append(compute_carbon_footprint(w_mv, co2_total, mv_y, portfolio_value, Y))

    # VW benchmark
    cf_vw.append(compute_vw_carbon_footprint(isins, co2_total, mv_y, Y))
    # WACI VW — TODO: compute vw_weights_annual and call compute_waci()
    waci_vw.append(np.nan)  # placeholder

    # Update portfolio value for next year (approx)
    # TODO: use actual cumulative MV returns to update V

print('Carbon metrics computed (MV portfolio and VW benchmark)')

In [ ]:
# TODO: identify top-10 carbon-intensive firms in the investment set
# Report firm names, ISIN codes, CI values, and portfolio weights

## 4. Section 3.2 — MV Portfolio with 50% Carbon Footprint Reduction

$$
\min_{\alpha_Y} \; \alpha_Y' \Sigma_Y \alpha_Y
\quad \text{s.t.} \quad
\text{CF}^{(p)}_Y \leq 0.5 \times \text{CF}^{(mv)}_{oos,Y},
\quad \alpha_Y'\mathbf{e}=1, \quad \alpha_i \geq 0
$$

In [ ]:
# TODO: roll solve_mv_carbon_constrained() over 2013–2024
# Budget = 0.5 * cf_mv[Y]

mv05_weights = {}  # year → pd.Series

print(f"{'Year':<6s} {'N':>5s} {'CF budget':>12s} {'Status':>12s}")
print('-' * 40)

for Y in years:
    isins = investment_sets[Y]
    N     = len(isins)
    win   = get_estimation_window(Y, ret_dates)
    R_mat = returns.loc[isins, win].values
    Sigma = pairwise_covariance(R_mat)

    cf_budget  = 0.5 * cf_mv[years.index(Y)]  # 50% of unconstrained MV CF
    c_vec      = _footprint_constraint_vector(isins, co2_total, mv_y, V0, Y)

    w, status  = solve_mv_carbon_constrained(Sigma, c_vec, cf_budget)
    mv05_weights[Y] = pd.Series(w, index=isins)
    print(f"{Y:<6d} {N:>5d} {cf_budget:>12.2f} {status:>12s}")

In [ ]:
# TODO: compute ex-post returns for P_mv(0.5)
# Hint: use compute_mv_returns() with mv05_weights

# mv05_ret = compute_mv_returns(investment_sets, mv05_weights, returns, ret_dates)

## 5. Section 3.3 — Tracking-Error Minimisation with 50% CF Reduction

$$
\min_{\alpha_Y} \; (\alpha_Y - \alpha^{(vw)}_Y)' \Sigma_Y (\alpha_Y - \alpha^{(vw)}_Y)
\quad \text{s.t.} \quad
\text{CF}^{(p)}_Y \leq 0.5 \times \text{CF}^{(vw)}_Y
$$

In [ ]:
# TODO: roll solve_te_carbon_constrained() over 2013–2024
# Need vw_weights_annual: dict year → np.array of VW weights for that year's investment set

vw05_weights = {}  # year → pd.Series

# for Y in years:
#     ...
#     w, status = solve_te_carbon_constrained(Sigma, vw_w, c_vec, cf_budget)
#     vw05_weights[Y] = pd.Series(w, index=isins)

In [ ]:
# TODO: compute ex-post returns for P_vw(0.5)
# vw05_ret = compute_mv_returns(investment_sets, vw05_weights, returns, ret_dates)

## 6. Section 4.1 — Net-Zero Pathway (−10% per year)

$$
\text{CF}^{(p)}_Y \leq (1 - \theta)^{Y - Y_0 + 1} \times \text{CF}^{(vw)}_{Y_0}
\quad \text{for } Y = 2013, \ldots, 2024
$$

with $\theta = 0.10$ and $Y_0 = 2013$.

In [ ]:
# Reference: VW carbon footprint at Y0
cf_vw_Y0 = cf_vw[years.index(Y0)]  # CF^(vw)_{Y0}

nz_weights = {}  # year → pd.Series

# for Y in years:
#     exponent   = Y - Y0 + 1
#     cf_budget  = (1 - THETA) ** exponent * cf_vw_Y0
#     ...
#     w, status  = solve_te_carbon_constrained(Sigma, vw_w, c_vec, cf_budget)
#     nz_weights[Y] = pd.Series(w, index=isins)

In [ ]:
# TODO: compute ex-post returns for P_vw(NZ)
# nz_ret = compute_mv_returns(investment_sets, nz_weights, returns, ret_dates)

## 7. Comparison of All Portfolios

### 7.1 Financial Performance

In [ ]:
# TODO: compute stats for all 5 portfolios and display summary table
# Portfolios: vw_ret, mv_ret, mv05_ret, vw05_ret, nz_ret

# Example:
# stats_all = pd.DataFrame({
#     'VW'       : compute_stats(vw_ret,   rf),
#     'MV'       : compute_stats(mv_ret,   rf),
#     'MV(0.5)'  : compute_stats(mv05_ret, rf),
#     'VW(0.5)'  : compute_stats(vw05_ret, rf),
#     'VW(NZ)'   : compute_stats(nz_ret,   rf),
# })
# print(stats_all.applymap(lambda x: f'{x:.4f}').to_string())

### 7.2 Cumulative Return Plot

In [ ]:
# TODO: plot cumulative returns for all portfolios
#
# fig = plot_cumulative_returns(
#     series_dict = {
#         'VW'     : vw_ret,
#         'MV'     : mv_ret,
#         'MV(0.5)': mv05_ret,
#         'VW(0.5)': vw05_ret,
#         'VW(NZ)' : nz_ret,
#     },
#     title     = 'Cumulative Performance: All Portfolios (2014–2025)',
#     save_path = FIGURES_DIR + 'cumulative_returns_all.png',
# )
# plt.show()

### 7.3 Carbon Footprint Evolution

In [ ]:
# TODO: plot CF over years for all strategies
#
# fig = plot_carbon_metric(
#     metric_dict = {
#         'VW'     : cf_vw,
#         'MV'     : cf_mv,
#         'MV(0.5)': cf_mv05,
#         'VW(0.5)': cf_vw05,
#         'VW(NZ)' : cf_nz,
#     },
#     years     = years,
#     ylabel    = 'Carbon Footprint (tCO2e / $M invested)',
#     title     = 'Carbon Footprint Evolution (2013–2024)',
#     save_path = FIGURES_DIR + 'carbon_footprint_evolution.png',
# )
# plt.show()

## 8. Portfolio Composition Analysis

Key questions to address:
- Which sectors are **over/under-weighted** in the carbon-constrained portfolios?
- Which firms are **excluded** when the CF constraint is binding?
- How does the **active share** evolve over time?

In [ ]:
# TODO: sector analysis
# Merge weights with static (country / sector info) and compare across portfolios

## 9. Summary Tables Export

In [ ]:
# TODO: export all summary tables to TABLES_DIR
# - summary_stats_part2.csv   (financial stats for all 5 portfolios)
# - carbon_footprint.csv      (CF per year per portfolio)
# - waci.csv                  (WACI per year per portfolio)
# - top10_high_carbon.csv     (top-10 carbon-intensive firms)